# Preparação dos dados
_Feature Engineering_

---

## Sumário

1. **Importação de bibliotecas**
2. **Carregamento das bases**
3. **Feature Engineering**
    - 3.1. Ajustando a tipagem das variáveis
    - 3.2. Criando novas features
4. **Salvando os DataFrames em formato parquet**

<br>

---

<br>

## 1. Importação de bibliotecas

In [1]:
# Importação de pacotes e definição de parâmetros globais

import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import warnings
import gc

from pathlib import Path

In [2]:
# Configuração de parâmetros globais

# Configurar para exibir todas as colunas do Dataframe
pd.set_option('display.max_columns', None)

# Configurar para exibir o conteúdo completo das colunas
pd.set_option('display.max_colwidth', None)

# Ignorar warnings para uma execução mais limpa
warnings.filterwarnings('ignore')

## 2. Carregamento das bases

In [3]:
# Efetuando a limpeza da memória antes do carregamento dos dados
print(f'\nQuantidade de objetos removidos da memória: {gc.collect()}')


Quantidade de objetos removidos da memória: 0


In [4]:
# Caminho para o diretório de dados
DATA_DIR = Path('../data/processed')

# Carregando os dados
try:
    table = pq.read_table(DATA_DIR)
    df = table.to_pandas()
except Exception as e:
    print(f'Erro ao carregar os dados: {e}')

In [5]:
print('VOLUMETRIA')
print(f'Quantidade de linhas (registros):  {df.shape[0]:,}')
print(f'Quantidade de colunas (variáveis): {df.shape[1]:,}')

VOLUMETRIA
Quantidade de linhas (registros):  10,000
Quantidade de colunas (variáveis): 24


In [6]:
df.head(10)

,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,age_bucket,tenure_bucket,balance_to_salary_ratio,products_per_tenure,is_high_credit_score,is_senior_new_customer,avg_salary_by_geography,customer_count_by_geography,is_zero_balance,is_multi_product_customer,is_active_with_card,Exited,processing_date
0,15565714,601,France,Male,47,1,64430.06,2,0,1,96517.97,Adult,New customer,0.667545,2.000000,0,0,99899.180814,5014,0,1,0,0,2026-03-30
1,15565806,532,France,Male,38,9,0.00,2,0,0,30583.95,Adult,Long term,0.000000,0.222222,0,0,99899.180814,5014,1,1,0,0,2026-03-30
2,15565879,845,France,Female,28,9,0.00,2,1,1,56185.98,Young,Long term,0.000000,0.222222,1,0,99899.180814,5014,1,1,1,0,2026-03-30
3,15565891,709,France,Male,39,8,0.00,2,1,0,56214.09,Adult,Long term,0.000000,0.250000,0,0,99899.180814,5014,1,1,0,0,2026-03-30
4,15565996,653,France,Male,44,8,0.00,2,1,1,154639.72,Adult,Long term,0.000000,0.250000,0,0,99899.180814,5014,1,1,1,0,2026-03-30
5,15566111,596,France,Male,39,9,0.00,1,1,0,48963.59,Adult,Long term,0.000000,0.111111,0,0,99899.180814,5014,1,0,0,0,2026-03-30
6,15566139,526,France,Female,37,5,53573.18,1,1,0,62830.97,Adult,Mid tenure,0.852656,0.200000,0,0,99899.180814,5014,0,0,0,0,2026-03-30
7,15566251,618,France,Female,37,5,96652.86,1,1,0,98686.40,Adult,Mid tenure,0.979394,0.200000,0,0,99899.180814,5014,0,0,0,1,2026-03-30
8,15566269,787,France,Male,25,5,0.00,2,1,0,47307.90,Young,Mid tenure,0.000000,0.400000,1,0,99899.180814,5014,1,1,0,0,2026-03-30
9,15566295,761,France,Female,33,6,138053.79,2,1,0,148779.41,Adult,Mid tenure,0.927909,0.333333,1,0,99899.180814,5014,0,1,0,0,2026-03-30


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   CustomerId                   10000 non-null  int32   
 1   CreditScore                  10000 non-null  int32   
 2   Geography                    10000 non-null  str     
 3   Gender                       10000 non-null  str     
 4   Age                          10000 non-null  int32   
 5   Tenure                       10000 non-null  int32   
 6   Balance                      10000 non-null  float64 
 7   NumOfProducts                10000 non-null  int32   
 8   HasCrCard                    10000 non-null  int32   
 9   IsActiveMember               10000 non-null  int32   
 10  EstimatedSalary              10000 non-null  float64 
 11  age_bucket                   10000 non-null  str     
 12  tenure_bucket                10000 non-null  str     
 13  balance_to_sa

## 3. Feature Engineering

### 3.1. Features criadas na construção da ABT (ETL)

In [8]:
# age_bucket - classificação da idade

df_temp = df[['CustomerId', 'Age', 'age_bucket']]
df_temp.groupby('age_bucket').sample(1)

,CustomerId,Age,age_bucket
7487,15752312,49,Adult
8566,15780124,74,Senior
9364,15800215,25,Young


In [9]:
# tenure_bucket - classificação do tempo de relacionamento

df_temp = df[['CustomerId', 'Tenure', 'tenure_bucket']]
df_temp.groupby('tenure_bucket').sample(1)

,CustomerId,Tenure,tenure_bucket
3022,15642291,8,Long term
772,15586300,7,Mid tenure
2335,15625092,3,New customer


In [ ]:
# balance_to_salary_ratio - relação entre o saldo e o salário estimado

df_temp = df[['CustomerId', 'Tenure', 'EstimatedSalary', 'balance_to_salary_ratio']]
df_temp.groupby('balance_to_salary_ratio')
df_temp.head(5)

,CustomerId,Tenure,EstimatedSalary,balance_to_salary_ratio
0,15565714,1,96517.97,0.667545
1,15565806,9,30583.95,0.000000
2,15565879,9,56185.98,0.000000
3,15565891,8,56214.09,0.000000
4,15565996,8,154639.72,0.000000


In [ ]:
# products_per_tenure - quantidade de produtos por tempo de relacionamento

df_temp = df[['CustomerId', 'NumOfProducts', 'Tenure', 'products_per_tenure']]
df_temp.groupby('products_per_tenure')
df_temp.head(5)

,CustomerId,NumOfProducts,Tenure,products_per_tenure
0,15565714,2,1,2.000000
1,15565806,2,9,0.222222
2,15565879,2,9,0.222222
3,15565891,2,8,0.250000
4,15565996,2,8,0.250000


In [ ]:
# is_high_credit_score - indicador de score de crédito alto, acima de 721

df_temp = df[['CustomerId', 'CreditScore', 'is_high_credit_score']]
df_temp.groupby('is_high_credit_score').sample(1)

,CustomerId,CreditScore,is_high_credit_score
4772,15685536,552,0
6730,15733872,791,1


In [ ]:
# is_senior_new_customer - inindicador se o novo cliente é idoso

df_temp = df[['CustomerId', 'Age', 'Tenure', 'is_senior_new_customer']]
df_temp.groupby('is_senior_new_customer').sample(1)

,CustomerId,Age,Tenure,is_senior_new_customer
8901,15788539,34,3,0
9110,15793726,79,0,1


In [ ]:
# avg_salary_by_geography - média salarial por região geográfica

df_temp = df[['CustomerId', 'EstimatedSalary', 'Geography', 'avg_salary_by_geography']]
df_temp.groupby('avg_salary_by_geography').sample(1)

,CustomerId,EstimatedSalary,Geography,avg_salary_by_geography
3645,15657105,195192.40,Spain,99440.572281
9624,15806967,77867.23,France,99899.180814
833,15586970,22485.64,Germany,101113.435102


In [ ]:
# customer_count_by_geography - quantidade de clientes por região geográfica

df_temp = df[['CustomerId', 'Geography', 'customer_count_by_geography']]
df_temp.groupby('customer_count_by_geography').sample(1)

,CustomerId,Geography,customer_count_by_geography
5244,15696371,Spain,2477
9334,15799217,Germany,2509
9050,15792029,France,5014


In [ ]:
# is_zero_balance - indicador de saldo zero

df_temp = df[['CustomerId', 'Balance', 'is_zero_balance']]
df_temp.groupby('is_zero_balance').sample(1)

,CustomerId,Balance,is_zero_balance
9419,15801462,109578.04,0
2252,15623277,0.00,1


In [ ]:
# is_multi_product_customer - indicador de cliente com múltiplos produtos contratados

df_temp = df[['CustomerId', 'NumOfProducts', 'is_multi_product_customer']]
df_temp.groupby('is_multi_product_customer').sample(1)

,CustomerId,NumOfProducts,is_multi_product_customer
6218,15721921,1,0
6484,15727391,2,1


In [ ]:
# is_active_with_card - indicador de cliente ativo com cartão de crédito

df_temp = df[['CustomerId', 'IsActiveMember', 'HasCrCard', 'is_active_with_card']]
df_temp.groupby('is_active_with_card').sample(1)

,CustomerId,IsActiveMember,HasCrCard,is_active_with_card
2046,15617320,0,1,0
9774,15809837,1,1,1
